In [ ]:
%matplotlib inline
from pathlib import Path
import hashlib
import json
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib import pyplot as plt
from matplotlib import patheffects
from matplotlib.text import Text
from matplotlib.cm import ScalarMappable
from matplotlib.collections import PathCollection
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.lines import Line2D
from scipy.stats import linregress
from probeinterface import read_openephys

from Utils.direction_comparison import load_hd_profiles, peak_angles
from Utils.plotting import LIGHT_PLOT_STYLE
from Utils.rflocate import load_rfmap, load_rf, rf_result_path
from Utils.si_utils import WaveformArtifactStore

# The existing probe plotting module sets rcParams during import.
with plt.rc_context():
    from Utils.probe_plotting import WaveformUnitPlotCollection

plt.rcParams.update(LIGHT_PLOT_STYLE)
root = Path("/mnt/senzailab/Kai/#Recording")
output_dir = Path("output/retinotopography")
output_dir.mkdir(parents=True, exist_ok=True)

recordings = [
    dict(mouse="m14", date="260609", rf_session=3, hd_session=1, position_session=3, probe="A"),
    dict(mouse="m15", date="260630", rf_session=3, hd_session=1, position_session=3, probe="A"),
    dict(mouse="m15", date="260630", rf_session=3, hd_session=1, position_session=3, probe="B"),
    dict(mouse="m19", date="260827", rf_session=2, hd_session=9, position_session=9, probe="A"),
    dict(mouse="m20", date="260921", rf_session=2, hd_session=9, position_session=2, probe="A"),
    dict(mouse="m21", date="261006", rf_session=3, hd_session=9, position_session=9, probe="A"),

]
hd_class = 3
response_window_s = (0.0, 0.2)
require_rf_detection = False  # All HD cells with a non-flat measured RF map; optional saved 2-D RF gate.
pd.DataFrame(recordings)

In [ ]:
hd_profiles = {}
source_rows = []
for recording in recordings:
    mouse, date, probe = (recording[key] for key in ("mouse", "date", "probe"))
    session_dir = root / mouse / date / f"{date}_{recording['rf_session']}"
    data_dir = session_dir / "data"
    position_data_dir = root / mouse / date / f"{date}_{recording['position_session']}" / "data"
    tc_file = (root / mouse / date / f"{date}_{recording['hd_session']}" /
               "data" / "tuning_curves" / f"Probe{probe}" / "tuning_curves.tc")
    rf_file = (data_dir / "rfmapping/good/-100_400_1ms" / f"Probe{probe}" /
               f"regular_unitsSpikeCounts_{date}_{recording['rf_session']}.rfmap")
    key = (mouse, date, probe)
    hd_profiles[key] = load_hd_profiles(tc_file, probe=probe, bins=180,
                                        smoothing_deg=0, hd_class=hd_class)
    source_rows.append({**recording, "hd_file": str(tc_file), "rf_file": str(rf_file),
                       "waveform_dir": str(position_data_dir / "waveform" / f"Probe{probe}"),
                       "position_dir": str(position_data_dir / "spike_position" / f"Probe{probe}"),
                       "settings_file": str(session_dir / date / "Record Node 102/settings.xml"),
                       "hd_count": len(hd_profiles[key])})
sources = pd.DataFrame(source_rows)
sources.to_csv(output_dir / "sources.csv", index=False)
display(sources[["mouse", "date", "rf_session", "hd_session", "position_session", "probe", "hd_count"]])

In [ ]:
session_groups = sources.groupby(["mouse", "date", "rf_session"], sort=False)
n_sessions = session_groups.ngroups
ncols = min(2, n_sessions)
nrows = (n_sessions + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(6 * ncols, 4.5 * nrows),
                         squeeze=False, layout="constrained", facecolor="white")
for ax in axes.flat[n_sessions:]:
    ax.remove()
axes = axes.ravel()[:n_sessions]
for ax, ((mouse, date, rf_session), session_sources) in zip(axes, session_groups, strict=True):
    curves = []
    for source in session_sources.itertuples():
        table = hd_profiles[(source.mouse, source.date, source.probe)]
        table = table.copy()
        table.attrs = {}
        curves.append(table)
    table = pd.concat(curves)
    angles = np.asarray(table.columns) % 360
    preferred = peak_angles(table, table.index) % 360
    values = table.to_numpy()[np.argsort(preferred, kind="stable")]
    values = values / np.nanmax(values, axis=1, keepdims=True)
    image = ax.imshow(values[:, np.argsort(angles)], origin="lower", aspect="auto",
                      interpolation="nearest", extent=(0, 360, 0, len(table)),
                      cmap="viridis", vmin=0, vmax=1)
    probes = "+".join(session_sources.probe)
    ax.set(title=f"{mouse} {date}_{rf_session} / {probes} · {len(table)} HD cells",
           xlabel="HD direction (°; clockwise)", ylabel="Units sorted by preferred direction",
           xticks=[0, 90, 180, 270, 360])
fig.colorbar(image, ax=axes, label="Firing rate / each unit's maximum", shrink=0.85)
fig.savefig(output_dir / "hd_overview.png", dpi=180, facecolor="white", transparent=False)
plt.show()

In [ ]:
geometry_rows = []
position_rows = []
for source in sources.itertuples():
    store = WaveformArtifactStore(source.waveform_dir)
    if source.position_session != source.rf_session:
        rf_store = WaveformArtifactStore(root / source.mouse / source.date /
                                         f"{source.date}_{source.rf_session}" /
                                         "data" / "waveform" / f"Probe{source.probe}")
        assert store.manifest["canonical_source"] == rf_store.manifest["canonical_source"]
        for unit_id in set(store.unit_summaries) & set(rf_store.unit_summaries):
            unit, rf_unit = store.unit_summaries[unit_id], rf_store.unit_summaries[unit_id]
            np.testing.assert_allclose([unit.unit_x_um, unit.unit_y_um],
                                       [rf_unit.unit_x_um, rf_unit.unit_y_um], rtol=0, atol=0)
        np.testing.assert_array_equal(store.channel_locations, rf_store.channel_locations)
        np.testing.assert_array_equal(store.channel_shank_ids, rf_store.channel_shank_ids)
    setup = read_openephys(source.settings_file, stream_name=f"Probe{source.probe}")
    channel_order = np.argsort(setup.device_channel_indices)
    coordinates = setup.contact_positions[channel_order]
    shanks = setup.shank_ids[channel_order].astype(int)
    kilosort_dir = Path(store.manifest["session"]["kilosort_dir"])
    kilosort_coordinates = np.load(kilosort_dir / "channel_positions.npy")
    kilosort_shanks = np.load(kilosort_dir / "ops.npy", allow_pickle=True).item()["probe"]["kcoords"]
    np.testing.assert_allclose(coordinates, store.channel_locations, rtol=0, atol=1e-6)
    np.testing.assert_allclose(coordinates, kilosort_coordinates, rtol=0, atol=1e-6)
    np.testing.assert_array_equal(shanks, store.channel_shank_ids)
    np.testing.assert_array_equal(shanks, np.asarray(kilosort_shanks).astype(int))
    tips = np.asarray(setup.annotations["shank_tips"])
    position_manifest = json.loads((Path(source.position_dir) / "manifest.json").read_text())
    position_method = (position_manifest["spike_position"]["method"]
                       if "spike_position" in position_manifest
                       else position_manifest["canonical_source"]["method"])
    for shank in np.unique(shanks):
        channel_y = coordinates[shanks == shank, 1]
        tip_y = tips[shank, 1]
        assert tip_y < channel_y.min()
        geometry_rows.append(dict(mouse=source.mouse, date=source.date, probe=source.probe,
                                  shank=int(shank), part_number=setup.annotations["part_number"],
                                  min_channel_y_um=channel_y.min(), max_channel_y_um=channel_y.max(),
                                  tip_y_um=tip_y, deeper_is="smaller_y",
                                  max_coordinate_error_um=float(np.max(abs(coordinates - store.channel_locations))),
                                  position_method=position_method))
    for unit_id, unit in store.unit_summaries.items():
        position_rows.append(dict(mouse=source.mouse, date=source.date, probe=source.probe,
                                  unit_id=unit_id, shank=int(store.channel_shank_ids[unit.best_channel_index]),
                                  x_um=unit.unit_x_um, y_um=unit.unit_y_um,
                                  best_channel_id=unit.best_channel_id,
                                  best_channel_y_um=unit.best_channel_y_um))
geometry = pd.DataFrame(geometry_rows)
positions = pd.DataFrame(position_rows)
geometry.to_csv(output_dir / "depth_convention.csv", index=False)
display(geometry.round(3))

In [ ]:
unit_rows = []
identity = ["mouse", "date", "probe", "unit_id"]
for source in sources.itertuples():
    table = hd_profiles[(source.mouse, source.date, source.probe)]
    preferred = pd.Series(peak_angles(table, table.index) % 360, index=table.index)
    maps = load_rfmap(source.rf_file).to_firing_rate(reconstruct_presentations=True).mean_rate(
        *response_window_s, show_progress=False)
    maps_by_id = {rf_map.unit_id: rf_map for rf_map in maps}
    detected = load_rf(rf_result_path(source.rf_file))
    detected_ids = set(detected.unit_ids[np.any(detected.mask_2d, axis=(1, 2))].tolist())
    for (_, unit_id), hd_angle in preferred.items():
        row = dict(mouse=source.mouse, date=source.date, probe=source.probe, unit_id=int(unit_id),
                   hd_class=hd_class, hd_preferred_deg=hd_angle, rf_max_angle_deg=np.nan,
                   rf_max_y_deg=np.nan, rf_max_response=np.nan, rf_peak_ties=0,
                   rf_detected_2d=unit_id in detected_ids, rf_peak_status="unit_not_in_rf")
        if unit_id in maps_by_id:
            rf_map = maps_by_id[unit_id]
            response = rf_map.to_2d_array()
            finite_values = response[np.isfinite(response)]
            row["rf_peak_status"] = "flat_or_unmeasured"
            if finite_values.size and np.ptp(finite_values) > 0:
                y, x = np.unravel_index(np.nanargmax(response), response.shape)
                row.update(rf_max_angle_deg=float(rf_map.x_positions[x]),
                           rf_max_y_deg=float(rf_map.y_positions[y]),
                           rf_max_response=float(response[y, x]),
                           rf_peak_ties=int(np.sum(response == response[y, x])),
                           rf_peak_status="measured_peak")
        unit_rows.append(row)
units = pd.DataFrame(unit_rows).merge(positions, on=identity, how="left", validate="one_to_one")
units["depth_eligible"] = np.isfinite(units.y_um) & np.isfinite(units.hd_preferred_deg)
units["shank_label"] = [
    f"{m} {d}/{p} S{int(s)}" if pd.notna(s) else None
    for m, d, p, s in units[["mouse", "date", "probe", "shank"]].itertuples(index=False, name=None)
]
hd_depth = units.loc[units.depth_eligible]
origins = hd_depth.groupby("shank_label").y_um.max()
units["topmost_hd_y_um"] = units.shank_label.map(origins)
units["normalized_depth_um"] = units.topmost_hd_y_um - units.y_um
units["rf_depth_eligible"] = units.depth_eligible & units.rf_max_angle_deg.notna()
if require_rf_detection:
    units["rf_depth_eligible"] &= units.rf_detected_2d
hd_depth = units.loc[units.depth_eligible].copy()
rf_depth = units.loc[units.rf_depth_eligible].copy()
np.testing.assert_allclose(hd_depth.groupby("shank_label").normalized_depth_um.min(), 0)
assert (hd_depth.normalized_depth_um >= 0).all()
assert not units.duplicated(identity).any()
units.to_csv(output_dir / "unit_angles_depth.csv", index=False)
summary = units.groupby(["mouse", "date", "probe"], sort=False).agg(
    hd_cells=("unit_id", "size"), hd_with_depth=("depth_eligible", "sum"),
    hd_with_rf_peak_and_depth=("rf_depth_eligible", "sum"),
    saved_significant_rf=("rf_detected_2d", "sum"))
display(summary)
display(units.loc[~units.depth_eligible, identity + ["hd_preferred_deg", "rf_peak_status"]])
shank_summary = hd_depth.groupby("shank_label", sort=True).agg(
    n_hd=("unit_id", "size"), topmost_hd_y_um=("y_um", "max"),
    normalized_span_um=("normalized_depth_um", "max"))
shank_summary["n_rf"] = rf_depth.groupby("shank_label").size().reindex(shank_summary.index, fill_value=0)
shank_summary.to_csv(output_dir / "shank_summary.csv")
display(shank_summary.round(3))

In [ ]:
def place_unit_labels(ax, renderer):
    # Search the rendered axes for nearby labels without moving the probe points.
    points = ax.transData.transform([text.xy for text in ax.texts])
    bounds = ax.get_window_extent()
    pixel_scale = renderer.points_to_pixels(1)
    boxes = [Text.get_window_extent(text, renderer) for text in ax.texts]
    step_x = max(box.width for box in boxes) / pixel_scale + 2
    step_y = max(box.height for box in boxes) / pixel_scale + 2
    offsets = sorted(
        ((sx * dx, sy * dy)
         for dx in np.arange(4, bounds.width / pixel_scale, step_x)
         for dy in np.arange(4, bounds.height / pixel_scale, step_y)
         for sx, sy in ((1, 1), (1, -1), (-1, 1), (-1, -1))),
        key=lambda offset: offset[0] ** 2 + offset[1] ** 2)
    placed = []
    for text in ax.texts:
        for dx, dy in offsets:
            text.set_position((dx, dy))
            text.set_horizontalalignment("left" if dx > 0 else "right")
            text.set_verticalalignment("bottom" if dy > 0 else "top")
            text.update_positions(renderer)
            box = Text.get_window_extent(text, renderer).padded(pixel_scale)
            inside = bounds.contains(box.x0, box.y0) and bounds.contains(box.x1, box.y1)
            covers_point = ((points[:, 0] >= box.x0) & (points[:, 0] <= box.x1) &
                            (points[:, 1] >= box.y0) & (points[:, 1] <= box.y1)).any()
            if inside and not covers_point and not any(box.overlaps(other) for other in placed):
                placed.append(box)
                break
        else:
            raise ValueError(f"No non-overlapping position for unit label {text.get_text()}")


def plot_probe_locations(units, sources, output_dir, cohort_label="all"):
    position_color_bins = 12
    hd_position_cmap = plt.colormaps["rainbow"].resampled(position_color_bins)
    hd_position_norm = BoundaryNorm(np.linspace(0, 360, position_color_bins + 1),
                                    position_color_bins, clip=True)
    # Keep the same colors for equal angles on the signed RF and unsigned HD bars.
    rf_position_cmap = ListedColormap(
        np.roll(hd_position_cmap(np.arange(position_color_bins)), -position_color_bins // 2, axis=0),
        name="rainbow_12_signed_rf")
    rf_position_norm = BoundaryNorm(np.linspace(-180, 180, position_color_bins + 1),
                                    position_color_bins, clip=True)
    color_check_angles = np.arange(0, 360, 0.5)
    np.testing.assert_allclose(
        hd_position_cmap(hd_position_norm(color_check_angles)),
        rf_position_cmap(rf_position_norm((color_check_angles + 180) % 360 - 180)),
        rtol=0, atol=0)
    missing_rf_color = "#9ca3af"
    position_plot_counts = []
    position_offset_errors_um = []

    for (mouse, date, rf_session), session_sources in sources.groupby(
            ["mouse", "date", "rf_session"], sort=False):
        session_units = units.loc[units.mouse.eq(mouse) & units.date.eq(date)]
        assert len(session_units) == session_sources.hd_count.sum()
        assert np.isfinite(session_units[["x_um", "y_um", "hd_preferred_deg"]]).all().all()
        probe_groups = []
        lower_y, upper_y = [], []
        for source in session_sources.itertuples():
            group = session_units.loc[session_units.probe.eq(source.probe)].copy()
            plot_collection = WaveformUnitPlotCollection(source.waveform_dir, [])
            plot_collection.probe = read_openephys(
                source.settings_file, stream_name=f"Probe{source.probe}")
            tips = np.asarray(plot_collection.probe.annotations["shank_tips"])
            contact_y = plot_collection.store.channel_locations[:, 1]
            lower_y.extend([tips[:, 1].min(), contact_y.min(), group.y_um.min()])
            upper_y.extend([contact_y.max(), group.y_um.max()])
            for row in group.itertuples():
                summary = plot_collection.store.unit_summaries[row.unit_id]
                np.testing.assert_allclose([summary.unit_x_um, summary.unit_y_um],
                                           [row.x_um, row.y_um], rtol=0, atol=1e-6)
            probe_groups.append((source.probe, group, plot_collection))
        ylim = (min(lower_y) - 50, max(upper_y) + 100)

        figure_width = 12 if len(probe_groups) == 1 else 20
        probe_width = max(np.ptp(collection.store.channel_locations[:, 0]) + 140
                          for _, _, collection in probe_groups)
        panel_width = (figure_width / 2 - 1.2) / len(probe_groups)
        figure_height = (13 if len(probe_groups) > 1 else
                         min(13, 2.4 + panel_width * (ylim[1] - ylim[0]) / probe_width))
        figure = plt.figure(figsize=(figure_width, figure_height), dpi=220,
                             layout="constrained", facecolor="white")
        figure.suptitle(f"{mouse} · {date}_{rf_session} · {cohort_label} {len(session_units)} HD cells",
                       fontsize=16)
        subfigures = figure.subfigures(1, 2, wspace=0.07)
        plotted_by_panel = []
        position_axes = []
        panel_specs = [
            ("HD preferred direction", "hd_preferred_deg", hd_position_cmap,
             hd_position_norm, np.arange(0, 361, 60), "HD preferred direction (°; 30° bins)"),
            ("RF maximum-bin horizontal angle", "rf_max_angle_deg", rf_position_cmap,
             rf_position_norm, np.arange(-180, 181, 60), "RF maximum-bin angle (°; 30° bins)"),
        ]
        for subfigure, (title, angle_column, cmap, norm, ticks, colorbar_label) in zip(
                subfigures, panel_specs, strict=True):
            subfigure.set_facecolor("white")
            subfigure.suptitle(title, fontsize=13)
            axes = subfigure.subplots(1, len(probe_groups), squeeze=False, sharey=True)[0]
            plotted_keys = set()
            for column, (ax, (probe, group, plot_collection)) in enumerate(
                    zip(axes, probe_groups, strict=True)):
                angles = group[angle_column].to_numpy()
                angles = angles % 360 if angle_column == "hd_preferred_deg" else (angles + 180) % 360 - 180
                unit_values = dict(zip(group.unit_id.astype(int), angles, strict=True))
                result = plot_collection.plot_unit_locations(
                    ax=ax, unit_values=unit_values, cmap=cmap, norm=norm,
                    missing_color=missing_rf_color)
                assert len(result.plotted_unit_ids) == len(group)
                assert set(result.plotted_unit_ids) == set(unit_values)
                plotted_keys.update((probe, unit_id) for unit_id in result.plotted_unit_ids)
                offsets = np.concatenate([
                    np.asarray(artist.get_offsets()) for artist in ax.collections
                    if isinstance(artist, PathCollection)
                ])
                expected = group[["x_um", "y_um"]].to_numpy()
                offsets = offsets[np.lexsort((offsets[:, 1], offsets[:, 0]))]
                expected = expected[np.lexsort((expected[:, 1], expected[:, 0]))]
                np.testing.assert_allclose(offsets, expected, rtol=0, atol=1e-6)
                position_offset_errors_um.append(float(np.max(abs(offsets - expected))))
                for unit in group.itertuples():
                    text = ax.annotate(str(int(unit.unit_id)), (unit.x_um, unit.y_um),
                                       xytext=(4, 4), textcoords="offset points", fontsize=7,
                                       color="#222222", ha="left", va="bottom",
                                       arrowprops=dict(arrowstyle="-", color="#6b7280", lw=0.4,
                                                       shrinkA=2, shrinkB=4),
                                       path_effects=[patheffects.withStroke(linewidth=2, foreground="white")])
                    text.set_in_layout(False)
                position_axes.append(ax)
                contact_x = plot_collection.store.channel_locations[:, 0]
                ax.set(xlim=(min(contact_x.min(), group.x_um.min()) - 70,
                             max(contact_x.max(), group.x_um.max()) + 70),
                       ylim=ylim, title=f"Probe {probe} · {len(group)} HD cells")
                ax.set_aspect("equal", adjustable="box")
                ax.tick_params(colors="#222222")
                ax.set_ylabel("Probe y (µm; larger = shallower)" if column == 0 else "")
                if angle_column == "rf_max_angle_deg":
                    missing = int(group[angle_column].isna().sum())
                    if missing:
                        ax.legend(handles=[Line2D([], [], linestyle="none", marker="o",
                                                 markerfacecolor=missing_rf_color, markeredgecolor="white",
                                                 markersize=7, label=f"No RF peak (n = {missing})")],
                                  loc="upper right", fontsize=8, framealpha=1,
                                  facecolor="white", edgecolor="#d1d5db")
            colorbar = subfigure.colorbar(ScalarMappable(norm=norm, cmap=cmap),
                                         ax=list(axes), orientation="horizontal",
                                         ticks=ticks, shrink=0.85, pad=0.06, fraction=0.04, drawedges=True)
            colorbar.set_label(colorbar_label)
            colorbar.dividers.set_color("white")
            colorbar.dividers.set_linewidth(0.8)
            plotted_by_panel.append(plotted_keys)
        assert plotted_by_panel[0] == plotted_by_panel[1]
        assert len(plotted_by_panel[0]) == len(session_units)
        figure.supxlabel("Recorded probe region · native x/y in µm · identical HD cells in both panels",
                         fontsize=10)
        figure.canvas.draw()
        figure.set_layout_engine("none")  # Keep display-coordinate label placement stable on save.
        for ax in position_axes:
            place_unit_labels(ax, figure.canvas.get_renderer())
        figure_file = output_dir / f"probe_locations_{mouse}_{date}_{rf_session}.png"
        figure.canvas.draw()
        renderer = figure.canvas.get_renderer()
        label_overlaps = 0
        label_count = 0
        for ax in position_axes:
            boxes = [Text.get_window_extent(text, renderer) for text in ax.texts]
            label_count += len(boxes)
            label_overlaps += sum(box.overlaps(other)
                                  for i, box in enumerate(boxes) for other in boxes[i + 1:])
        assert label_count == 2 * len(session_units)
        assert label_overlaps == 0
        figure.savefig(figure_file, dpi=220, facecolor="white", transparent=False)
        position_plot_counts.append(dict(
            unit_id_labels=label_count, unit_id_label_overlaps=int(label_overlaps),
            mouse=mouse, date=date, rf_session=int(rf_session), panels=2,
            probe_axes_per_panel=len(probe_groups), hd_cells_per_panel=len(session_units),
            rf_colored_cells=int(session_units.rf_max_angle_deg.notna().sum()),
            rf_gray_cells=int(session_units.rf_max_angle_deg.isna().sum()),
            plotted_units_by_probe={probe: group.unit_id.astype(int).tolist()
                                   for probe, group, _ in probe_groups},
            position_units="um", y_direction="larger_y_is_shallower",
            colormap="rainbow", color_bins=position_color_bins, color_bin_width_deg=360 / position_color_bins,
            figure=str(figure_file)))
        plt.show()
    return position_plot_counts, position_offset_errors_um


position_plot_counts, position_offset_errors_um = plot_probe_locations(
    units, sources, output_dir)


## One 1×2 depth figure per session

Each requested session has its own figure: **left, normalized depth vs HD preferred direction; right, normalized depth vs RF maximum-bin horizontal angle**. All eligible shanks from that session share each panel, with the same shank colors on both sides. m15 Probe A and B appear together, distinguished by probe + shank. Colored lines are per-shank OLS fits; the dashed black line is the **session's** pooled descriptive fit. Fits require at least 3 units and 2 distinct depths. Unfit shanks remain visible as points; empty shanks are omitted.

The regression CSV contains per-shank and per-session sample size, slope (°/µm), intercept, R², p value, and standard error. There is no all-session pooled fit.

**Angles are circular.** These requested OLS fits use HD in [0, 360) and RF in its native signed coordinate interval, without data-dependent unwrapping. A slope/p value can change when the angular origin is moved; it is descriptive and is not a circular-linear test. Session-pooled p values also treat units as independent and do not account for shank clustering. Inspect the colored within-shank relationships rather than inferring a population effect from the pooled p value alone.


In [ ]:
def fit_relationship(data, angle_column, relationship):
    rows = []
    groups = [("pooled", data), *data.groupby("shank_label", sort=True)]
    for label, group in groups:
        row = dict(relationship=relationship, shank=label, n=len(group),
                   slope_deg_per_um=np.nan, intercept_deg=np.nan, r_squared=np.nan,
                   p_value=np.nan, slope_stderr=np.nan, status="insufficient_distinct_depths_or_units")
        if len(group) >= 3 and group.normalized_depth_um.nunique() >= 2:
            result = linregress(group.normalized_depth_um, group[angle_column])
            row.update(slope_deg_per_um=result.slope, intercept_deg=result.intercept,
                       r_squared=result.rvalue**2, p_value=result.pvalue,
                       slope_stderr=result.stderr, status="fit")
        rows.append(row)
    return pd.DataFrame(rows)


def plot_depth(ax, data, angle_column, fits, colors, title, angle_label, ylim):
    by_shank = fits.set_index("shank")
    for label, group in data.groupby("shank_label", sort=True):
        color = colors[label]
        probe_shank = f"{group.probe.iloc[0]} S{int(group.shank.iloc[0])}"
        ax.scatter(group.normalized_depth_um, group[angle_column], s=32, color=color,
                   edgecolor="white", linewidth=0.35, alpha=0.9,
                   label=f"{probe_shank} (n={len(group)})")
        fit = by_shank.loc[label]
        if fit.status == "fit":
            x = np.array([group.normalized_depth_um.min(), group.normalized_depth_um.max()])
            ax.plot(x, fit.intercept_deg + fit.slope_deg_per_um * x, color=color, linewidth=1.1)
    fit = by_shank.loc["pooled"]
    if fit.status == "fit":
        x = np.array([data.normalized_depth_um.min(), data.normalized_depth_um.max()])
        ax.plot(x, fit.intercept_deg + fit.slope_deg_per_um * x, "k--", linewidth=1.8,
                label="Session OLS")
        title += f"\nSession OLS: slope={fit.slope_deg_per_um:.3f}°/µm · R²={fit.r_squared:.3f} · p={fit.p_value:.3g}"
    ax.set(title=title, xlabel="Normalized depth (µm below topmost HD unit)",
           ylabel=angle_label, ylim=ylim)
    ax.grid(alpha=0.45)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.16), ncol=3, fontsize=9,
              frameon=True, borderaxespad=0)


In [ ]:
regression_tables = []
session_plot_counts = []
for (mouse, date, rf_session), session_sources in sources.groupby(
        ["mouse", "date", "rf_session"], sort=False):
    hd_session = hd_depth.loc[hd_depth.mouse.eq(mouse) & hd_depth.date.eq(date)]
    rf_session_units = rf_depth.loc[rf_depth.mouse.eq(mouse) & rf_depth.date.eq(date)]
    shank_labels = sorted(hd_session.shank_label.unique())
    colors = dict(zip(shank_labels, plt.colormaps["tab10"](np.arange(len(shank_labels))), strict=True))
    hd_fits = fit_relationship(hd_session, "hd_preferred_deg", "HD")
    rf_fits = fit_relationship(rf_session_units, "rf_max_angle_deg", "RF")
    regression_tables.append(pd.concat([hd_fits, rf_fits], ignore_index=True).assign(
        mouse=mouse, date=date, rf_session=int(rf_session)))

    fig, axes = plt.subplots(1, 2, figsize=(14, 7), sharex=True, layout="constrained")
    probes = "+".join(session_sources.probe)
    fig.suptitle(f"{mouse} · {date}_{rf_session} · Probe {probes}", fontsize=16)
    plot_depth(axes[0], hd_session, "hd_preferred_deg", hd_fits, colors,
               f"HD preferred direction · {len(hd_session)} HD cells",
               "HD preferred direction (°; clockwise)", (-5, 365))
    plot_depth(axes[1], rf_session_units, "rf_max_angle_deg", rf_fits, colors,
               f"RF maximum-bin angle · {len(rf_session_units)} HD cells",
               "RF maximum spatial bin: horizontal angle (°)", (-185, 185))
    axes[0].set_yticks([0, 90, 180, 270, 360])
    axes[1].set_yticks([-180, -90, 0, 90, 180])
    figure_file = output_dir / f"session_{mouse}_{date}_{rf_session}.png"
    fig.savefig(figure_file, dpi=180, facecolor="white", transparent=False)
    session_plot_counts.append(dict(mouse=mouse, date=date, rf_session=int(rf_session),
                                    hd_cells=len(hd_session), rf_cells=len(rf_session_units),
                                    hd_shanks=len(shank_labels), rf_shanks=rf_session_units.shank_label.nunique(),
                                    panels=len(fig.axes), figure=str(figure_file)))
    plt.show()
regressions = pd.concat(regression_tables, ignore_index=True)
regressions.to_csv(output_dir / "linear_regressions.csv", index=False)
display(regressions.loc[regressions.shank.eq("pooled"),
                       ["mouse", "date", "rf_session", "relationship", "n",
                        "slope_deg_per_um", "r_squared", "p_value"]].round(5))


## Verification and saved outputs

The checks below verify that HD units were joined once, the topmost HD unit on each nonempty shank has zero depth, RF selection did not move those references, and all finite plotted data come from a Class 3 unit. Geometry verification covers every recorded channel on every shank, including empty shanks. Empty shanks are absent from the plots and regression tables.

Outputs: `hd_overview.png`, `probe_locations_<mouse>_<date>_<rf_session>.png` position figures (one per requested RF session, each 1×2, with all HD cells in both panels), `session_<mouse>_<date>_<rf_session>.png` figures (one per requested RF session, each 1×2), `sources.csv`, `depth_convention.csv`, `unit_angles_depth.csv`, `shank_summary.csv`, `linear_regressions.csv`, and `verification.json` in `output/retinotopography/`.

In [ ]:
assert units.hd_class.eq(3).all()
assert len(units) == sources.hd_count.sum()
assert np.isfinite(hd_depth[["normalized_depth_um", "hd_preferred_deg"]]).all().all()
assert np.isfinite(rf_depth[["normalized_depth_um", "rf_max_angle_deg"]]).all().all()
assert set(rf_depth.shank_label) <= set(hd_depth.shank_label)
np.testing.assert_allclose(hd_depth.normalized_depth_um,
                           hd_depth.groupby("shank_label").y_um.transform("max") - hd_depth.y_um)
np.testing.assert_allclose(rf_depth.topmost_hd_y_um,
                           rf_depth.shank_label.map(hd_depth.groupby("shank_label").y_um.max()))
assert geometry.max_coordinate_error_um.eq(0).all()
expected_sessions = sources.groupby(["mouse", "date", "rf_session"]).ngroups
assert len(session_plot_counts) == expected_sessions
assert all(item["panels"] == 2 for item in session_plot_counts)
assert sum(item["hd_cells"] for item in session_plot_counts) == len(hd_depth)
assert sum(item["rf_cells"] for item in session_plot_counts) == len(rf_depth)
assert len(position_plot_counts) == expected_sessions
assert sum(item["hd_cells_per_panel"] for item in position_plot_counts) == len(units)
assert all(item["rf_colored_cells"] + item["rf_gray_cells"] == item["hd_cells_per_panel"]
           for item in position_plot_counts)
assert max(position_offset_errors_um) <= 1e-6
source_hashes = {}
for source in sources.itertuples():
    for filename in [source.hd_file, source.settings_file,
                     str(Path(source.waveform_dir) / "channels.csv"),
                     str(Path(source.position_dir) / "positions.csv")]:
        source_hashes[filename] = hashlib.sha256(Path(filename).read_bytes()).hexdigest()
verification = dict(
    python_executable=sys.executable,
    hd_class=hd_class, response_window_s=response_window_s,
    require_rf_detection=require_rf_detection, hd_cells=int(len(units)),
    hd_cells_with_depth=int(len(hd_depth)), rf_cells_with_depth=int(len(rf_depth)),
    plotted_hd_shanks=int(hd_depth.shank_label.nunique()),
    plotted_rf_shanks=int(rf_depth.shank_label.nunique()),
    depth_origin="topmost_hd_per_shank", depth_positive_direction="toward_tip",
    all_shanks_smaller_y_is_deeper=bool(geometry.deeper_is.eq("smaller_y").all()),
    max_channel_coordinate_error_um=float(geometry.max_coordinate_error_um.max()),
    session_figures=session_plot_counts, probe_position_figures=position_plot_counts,
    max_probe_position_offset_error_um=max(position_offset_errors_um), sha256=source_hashes)
(output_dir / "verification.json").write_text(json.dumps(verification, indent=2) + "\n")
display(pd.Series({k: v for k, v in verification.items() if k != "sha256"}))
display(regressions.loc[regressions.shank.eq("pooled")].round(6))


## ADN selected from the annotated probe plots

The explicit unit list is in `config/retinotopography_adn_units.json`. For m14, m15,
and m20, take the union of the HD and RF panel circles in `scan113026.pdf`.
For m19, the new larger contour in `IMG_9872.HEIC` replaces both old PDF contours;
the smaller old contour still visible in the photo is not an additional selection.
Include points touching the boundary. ADN is the user's anatomical annotation.

Use the archived 270-unit table in `config/retinotopography_adn/` that produced
the annotated plots, rather than reload subsequently changed tuning-curve files.
The selection config records the original PDF/image paths and SHA256 hashes;
those binary originals remain external. The included tables and explicit unit
IDs reproduce the prior derived selection without repeating anatomical validation. Native HD/RF angles and probe
coordinates stay unchanged. Select units first, then set zero at the highest
**selected HD** unit on each shank: `depth = max(selected HD y) - y`. RF uses the
same HD origin. Unit IDs are shown on the physical probe plots only.

Repeat the existing per-shank and session-pooled OLS. These are descriptive linear
fits to native circular angles (no unwrapping); p values are nominal, depend on
the angular seam, and do not account for shank clustering or multiple tests.
Results and source snapshots are saved under `output/retinotopography/adn/`.


In [ ]:
adn_config_path = Path("config/retinotopography_adn_units.json")
adn_config = json.loads(adn_config_path.read_text())
adn_output_dir = Path("output/retinotopography/adn")
adn_output_dir.mkdir(parents=True, exist_ok=True)
for source_key in ("source_unit_table", "source_recordings", "source_verification"):
    assert hashlib.sha256(Path(adn_config[source_key]).read_bytes()).hexdigest() ==                 adn_config[f"{source_key}_sha256"]
adn_source_units = pd.read_csv(adn_config["source_unit_table"], dtype={"date": str})
adn_sources = pd.read_csv(adn_config["source_recordings"], dtype={"date": str})
adn_identity = ["mouse", "date", "probe", "unit_id"]
adn_selection = pd.DataFrame([
    dict(mouse=record["mouse"], date=record["date"], probe=record["probe"], unit_id=unit_id,
         selection_source=record.get("replacement_image", "scan113026.pdf"),
         boundary_included=unit_id in record["boundary_unit_ids"])
    for record in adn_config["recordings"] for unit_id in record["unit_ids"]
])
assert not adn_selection.duplicated(adn_identity).any()
assert not adn_source_units.duplicated(adn_identity).any()
assert set(map(tuple, adn_selection[adn_identity].to_numpy())) <=             set(map(tuple, adn_source_units[adn_identity].to_numpy()))
adn_audit = adn_source_units.merge(adn_selection, on=adn_identity, how="left",
                                    sort=False, validate="one_to_one")
adn_audit["adn_selected"] = adn_audit.selection_source.notna()
adn_audit.to_csv(adn_output_dir / "selection_audit.csv", index=False)
adn_units = adn_audit.loc[adn_audit.adn_selected].copy()
assert len(adn_units) == len(adn_selection)


In [ ]:
# Retain the original origin so both depth conventions remain auditable.
adn_units["all_hd_topmost_y_um"] = adn_units.topmost_hd_y_um
adn_units["all_hd_normalized_depth_um"] = adn_units.normalized_depth_um
adn_origins = adn_units.loc[adn_units.depth_eligible].groupby("shank_label").y_um.max()
adn_units["topmost_hd_y_um"] = adn_units.shank_label.map(adn_origins)
adn_units["normalized_depth_um"] = adn_units.topmost_hd_y_um - adn_units.y_um
adn_hd_depth = adn_units.loc[adn_units.depth_eligible].copy()
adn_rf_depth = adn_units.loc[adn_units.rf_depth_eligible].copy()
adn_units.to_csv(adn_output_dir / "unit_angles_depth.csv", index=False)
adn_summary = adn_units.groupby(["mouse", "date", "probe"], sort=False).agg(
    hd_cells=("unit_id", "size"), hd_with_depth=("depth_eligible", "sum"),
    hd_with_rf_peak_and_depth=("rf_depth_eligible", "sum"),
    saved_significant_rf=("rf_detected_2d", "sum"))
adn_summary.to_csv(adn_output_dir / "probe_summary.csv")
adn_sources["hd_count"] = [
    int(adn_summary.loc[(row.mouse, row.date, row.probe), "hd_cells"])
    for row in adn_sources.itertuples()
]
adn_shank_summary = adn_hd_depth.groupby("shank_label", sort=True).agg(
    n_hd=("unit_id", "size"), topmost_hd_y_um=("y_um", "max"),
    normalized_span_um=("normalized_depth_um", "max"))
adn_shank_summary["n_rf"] = adn_rf_depth.groupby("shank_label").size().reindex(
    adn_shank_summary.index, fill_value=0)
adn_shank_summary.to_csv(adn_output_dir / "shank_summary.csv")
display(adn_summary)
display(adn_shank_summary.round(3))


In [ ]:
adn_fit_tables = []
for (mouse, date, rf_session), session_sources in adn_sources.groupby(
        ["mouse", "date", "rf_session"], sort=False):
    hd_session = adn_hd_depth.loc[adn_hd_depth.mouse.eq(mouse) & adn_hd_depth.date.eq(date)]
    rf_session_units = adn_rf_depth.loc[adn_rf_depth.mouse.eq(mouse) & adn_rf_depth.date.eq(date)]
    hd_fits = fit_relationship(hd_session, "hd_preferred_deg", "HD")
    rf_fits = fit_relationship(rf_session_units, "rf_max_angle_deg", "RF")
    adn_fit_tables.append(pd.concat([hd_fits, rf_fits], ignore_index=True).assign(
        mouse=mouse, date=date, rf_session=int(rf_session)))
adn_regressions = pd.concat(adn_fit_tables, ignore_index=True)
adn_regressions.to_csv(adn_output_dir / "linear_regressions.csv", index=False)
display(adn_regressions.loc[adn_regressions.shank.eq("pooled"),
                           ["mouse", "date", "relationship", "n", "slope_deg_per_um",
                            "r_squared", "p_value"]].round(5))


In [ ]:
adn_session_plot_counts = []
for (mouse, date, rf_session), session_sources in adn_sources.groupby(
        ["mouse", "date", "rf_session"], sort=False):
    hd_session = adn_hd_depth.loc[adn_hd_depth.mouse.eq(mouse) & adn_hd_depth.date.eq(date)]
    rf_session_units = adn_rf_depth.loc[adn_rf_depth.mouse.eq(mouse) & adn_rf_depth.date.eq(date)]
    fits = adn_regressions.loc[adn_regressions.mouse.eq(mouse) & adn_regressions.date.eq(date)]
    # Keep each shank's color identical to the original full-cohort plot.
    original_session = adn_source_units.loc[
        adn_source_units.mouse.eq(mouse) & adn_source_units.date.eq(date)]
    shank_labels = sorted(original_session.shank_label.dropna().unique())
    colors = dict(zip(shank_labels, plt.colormaps["tab10"](np.arange(len(shank_labels))), strict=True))
    with plt.rc_context(LIGHT_PLOT_STYLE):
        fig, axes = plt.subplots(1, 2, figsize=(14, 7), sharex=True, layout="constrained",
                                 facecolor="white")
        probes = "+".join(session_sources.probe)
        fig.suptitle(f"ADN · {mouse} · {date}_{rf_session} · Probe {probes}", fontsize=16)
        plot_depth(axes[0], hd_session, "hd_preferred_deg", fits.loc[fits.relationship.eq("HD")],
                   colors, f"HD preferred direction · {len(hd_session)} selected HD cells",
                   "HD preferred direction (°; clockwise)", (-5, 365))
        plot_depth(axes[1], rf_session_units, "rf_max_angle_deg", fits.loc[fits.relationship.eq("RF")],
                   colors, f"RF maximum-bin angle · {len(rf_session_units)} selected HD cells",
                   "RF maximum spatial bin: horizontal angle (°)", (-185, 185))
        axes[0].set_yticks([0, 90, 180, 270, 360])
        axes[1].set_yticks([-180, -90, 0, 90, 180])
        for ax in axes:
            ax.set_facecolor("white")
            ax.set_xlabel("Depth (µm below topmost selected HD unit on each shank)")
        point_counts = [sum(len(artist.get_offsets()) for artist in ax.collections
                            if isinstance(artist, PathCollection)) for ax in axes]
        assert point_counts == [len(hd_session), len(rf_session_units)]
        figure_file = adn_output_dir / f"session_{mouse}_{date}_{rf_session}.png"
        fig.savefig(figure_file, dpi=180, facecolor="white", transparent=False)
        adn_session_plot_counts.append(dict(
            mouse=mouse, date=date, rf_session=int(rf_session), hd_cells=point_counts[0],
            rf_cells=point_counts[1], hd_shanks=int(hd_session.shank_label.nunique()),
            rf_shanks=int(rf_session_units.shank_label.nunique()), figure=str(figure_file)))
        plt.show()


### Selected units at their native probe positions

Only selected ADN units are shown. Both panels contain the same selected HD cells;
gray points have no RF peak. The labels are unit IDs, anchored to the unchanged
physical x/y coordinates. Probe y increases upward; the depth plots above increase
downward from the per-shank selected HD origin.


In [ ]:
with plt.rc_context(LIGHT_PLOT_STYLE):
    adn_position_plot_counts, adn_position_offset_errors_um = plot_probe_locations(
        adn_units, adn_sources, adn_output_dir, cohort_label="ADN ·")


In [ ]:
assert adn_units.hd_class.eq(3).all()
assert len(adn_units) == adn_sources.hd_count.sum()
assert len(adn_hd_depth) == len(adn_units)
assert set(adn_rf_depth.shank_label) <= set(adn_hd_depth.shank_label)
np.testing.assert_allclose(adn_hd_depth.groupby("shank_label").normalized_depth_um.min(), 0)
assert (adn_hd_depth.normalized_depth_um >= 0).all()
np.testing.assert_allclose(adn_hd_depth.normalized_depth_um,
                           adn_hd_depth.groupby("shank_label").y_um.transform("max") - adn_hd_depth.y_um)
np.testing.assert_allclose(adn_rf_depth.topmost_hd_y_um, adn_rf_depth.shank_label.map(adn_origins))
unchanged_columns = [column for column in adn_source_units.columns
                     if column not in adn_identity + ["topmost_hd_y_um", "normalized_depth_um"]]
pd.testing.assert_frame_equal(
    adn_units.set_index(adn_identity)[unchanged_columns],
    adn_source_units.set_index(adn_identity).loc[
        adn_units.set_index(adn_identity).index, unchanged_columns])
expected_adn_sessions = adn_sources.groupby(["mouse", "date", "rf_session"]).ngroups
assert len(adn_session_plot_counts) == len(adn_position_plot_counts) == expected_adn_sessions
assert sum(item["hd_cells"] for item in adn_session_plot_counts) == len(adn_hd_depth)
assert sum(item["rf_cells"] for item in adn_session_plot_counts) == len(adn_rf_depth)
assert sum(item["hd_cells_per_panel"] for item in adn_position_plot_counts) == len(adn_units)
assert max(adn_position_offset_errors_um) <= 1e-6
adn_source_hashes = {
    str(path): hashlib.sha256(Path(path).read_bytes()).hexdigest()
    for path in [adn_config_path, adn_config["source_unit_table"],
                 adn_config["source_recordings"], adn_config["source_verification"]]
}
adn_verification = dict(
    python_executable=sys.executable, analysis_source="archived unit table matching annotated plots",
    raw_tuning_curves_reloaded=False, region_label=adn_config["region_label"],
    recorded_annotation_evidence={key: {"path": adn_config[key], "sha256": adn_config[f"{key}_sha256"]}
                                  for key in ("source_pdf", "replacement_image")},
    selection_rule=adn_config["selection_rule"],
    m19_replacement_rule=next(record["selection_rule"] for record in adn_config["recordings"]
                              if record["mouse"] == "m19"),
    hd_cells=int(len(adn_units)), hd_cells_with_depth=int(len(adn_hd_depth)),
    rf_cells_with_depth=int(len(adn_rf_depth)), hd_shanks=int(adn_hd_depth.shank_label.nunique()),
    rf_shanks=int(adn_rf_depth.shank_label.nunique()), depth_origin="topmost_selected_hd_per_shank",
    depth_positive_direction="toward_tip", native_angles_and_coordinates_unchanged=True,
    max_probe_position_offset_error_um=max(adn_position_offset_errors_um),
    fit_method="same native-angle OLS as full cohort; nominal p values, no circular unwrap or cluster correction",
    session_figures=adn_session_plot_counts, probe_position_figures=adn_position_plot_counts,
    sha256=adn_source_hashes)
(adn_output_dir / "verification.json").write_text(json.dumps(adn_verification, indent=2) + "\n")
display(pd.Series({key: value for key, value in adn_verification.items()
                   if key not in ("session_figures", "probe_position_figures", "sha256")}))
